## Simple AI Agent (no memory, tools, or guardrails)
- Agent will evaluate the statement and verify it's accuracy.
- This agent doesn't have memory, tools or guardrails. It will just use the LLM to evaluate the statement and provide a final verdict and explanation.

### Environment setup - install required libraries

In [1]:
# Install the specific version of openai-agents
%pip uninstall -y openai-agents
%pip install --upgrade pydantic
%pip install --no-cache-dir openai-agents==0.2.2
%pip install python-dotenv langchain-openai==0.2.1

Found existing installation: openai-agents 0.2.2
Uninstalling openai-agents-0.2.2:
  Successfully uninstalled openai-agents-0.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 13.8 MB/s eta 0:00:00


In [2]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
import logging

# Import the Agent class to create and manage AI agents
from agents import Agent, Runner

# Import the Runner class, which is used to run an agent and get its output
# from agents import Runner

# Verification: inspect Runner.run signature and DEFAULT_AGENT_RUNNER attrs
import inspect

# This will be used to load the API key from the .env file
from dotenv import load_dotenv

In [3]:
load_dotenv()

# Get the OpenAI API key from environment variables or prompt if missing
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

# Configure the OpenAI Client using our key
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client successfully configured.")

OpenAI client successfully configured.


In [4]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

### Build and run AI agent using Open AI Agents SDK

In [5]:
# Define the instructions for the fact-checker AI Agent
fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
"""

In [ ]:
# Create a new agent called "Fact Checker"
fact_checker_agent = Agent(
    name = "Fact Checker",                      # Name of the agent
    instructions = fact_checker_instructions,   # The rules and behavior for the agent
    model = "gpt-4.1-mini"                      # The AI model (LLM) to use
    # model = "gpt-5-mini"
)

# Print a confirmation message that the agent was created
print(f"Agent '{fact_checker_agent.name}' created successfully!")

Agent 'Fact Checker' created successfully!


In [ ]:
# try:
#     from agents import Runner
#     from agents.run import DEFAULT_AGENT_RUNNER
#     print("Runner.run signature:")
#     print(inspect.signature(Runner.run))
#     print()
#     print("DEFAULT_AGENT_RUNNER type:", type(DEFAULT_AGENT_RUNNER))
#     print("DEFAULT_AGENT_RUNNER public attrs:")
#     print([a for a in dir(DEFAULT_AGENT_RUNNER) if not a.startswith('_')])
# except Exception as e:
#     print("Verification failed:", e)

In [ ]:
# Reduce noisy tracing/telemetry logs from the agents package (non-fatal errors)
logging.getLogger("openai.agents").setLevel(logging.WARNING)

# A statement we want the Fact Checker agent to verify
statement = "The Great Wall of China is visible from space with the naked eye."
# statement = "The tallest mountain in the world is Mount Everst"

# Display the statement we're going to check (in markdown format for nicer formatting)
print_markdown(f"Asking the Fact Checker to verify: '{statement}'")


# Run the agent without a custom run_config to avoid run_config attribute issues
try:
    response = await Runner.run(
        starting_agent = fact_checker_agent,
        input = statement,
    )
except Exception as e:
    logging.warning(f"Runner.run failed: {e}")
    # Re-raise so the notebook shows the full error for debugging
    raise

# Display the agent's response
print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)    # Shows the final verdict and explanation

Asking the Fact Checker to verify: 'The Great Wall of China is visible from space with the naked eye.'

❌ FALSE: The Great Wall of China is generally not visible from space with the naked eye due to its narrow width and the fact that it blends into the surrounding terrain.

### Check whether AI Agent can re-call the information (no memory) generated above in the same conversation 

In [8]:
check_recall_statement = "What did we discuss in the last message?"

response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)


🤖 Agent's Response:


❌ FALSE: The last message did not contain a statement to fact-check, but rather a question asking what was discussed previously.